# 4.2 Ridge Regression Model

This notebook trains a Ridge Regression model to predict charging station suitability scores for cities.

## Objectives:
1. Load processed data
2. Train Ridge Regression model with hyperparameter tuning
3. Evaluate model performance
4. Compare with Linear Regression
5. Save trained model


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Load processed data
X_train = np.load('../../data/processed/X_train.npy')
X_test = np.load('../../data/processed/X_test.npy')
y_train = np.load('../../data/processed/y_train.npy')
y_test = np.load('../../data/processed/y_test.npy')
feature_columns = joblib.load('../../data/processed/feature_columns.pkl')

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Features: {feature_columns}")


Training set shape: (716, 11)
Test set shape: (179, 11)
Features: ['EV_Count', 'Avg_Range', 'Range_Std', 'Avg_MSRP', 'Avg_Age', 'BEV_Ratio', 'PHEV_Ratio', 'Make_Diversity', 'Distance_From_Seattle', 'Distance_From_Spokane', 'County_Encoded']


In [3]:
# Train Ridge Regression Model
print("Training Ridge Regression Model...")

# Ensure targets are 1D numpy arrays
y_train = np.array(y_train).ravel()
y_test = np.array(y_test).ravel()

# Check for NaNs in targets and remove corresponding samples
train_nans = np.isnan(y_train).sum()
test_nans = np.isnan(y_test).sum()
print(f"y_train NaNs: {train_nans}, y_test NaNs: {test_nans}")

if train_nans > 0:
    print("Removing samples with NaN targets from training set...")
    mask = ~np.isnan(y_train)
    X_train = X_train[mask]
    y_train = y_train[mask]
    print(f"New training shape: {X_train.shape}, {y_train.shape}")

if test_nans > 0:
    print("Removing samples with NaN targets from test set...")
    mask = ~np.isnan(y_test)
    X_test = X_test[mask]
    y_test = y_test[mask]
    print(f"New test shape: {X_test.shape}, {y_test.shape}")

# Optionally save cleaned arrays for reproducibility (does not overwrite originals)
np.save('../../data/processed/X_train_clean.npy', X_train)
np.save('../../data/processed/y_train_clean.npy', y_train)
np.save('../../data/processed/X_test_clean.npy', X_test)
np.save('../../data/processed/y_test_clean.npy', y_test)
print("✓ Cleaned arrays saved to data/processed (clean copies)")

# Initialize and train the model with regularization
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train, y_train)

# Make predictions
y_train_pred = ridge_model.predict(X_train)
y_test_pred = ridge_model.predict(X_test)

print("✓ Model trained successfully")

# Calculate metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("\nModel Performance:")
print("=" * 30)
print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Training R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Training MAE: {train_mae:.4f}")
print(f"Test MAE: {test_mae:.4f}")


Training Ridge Regression Model...
y_train NaNs: 397, y_test NaNs: 92
Removing samples with NaN targets from training set...
New training shape: (319, 11), (319,)
Removing samples with NaN targets from test set...
New test shape: (87, 11), (87,)
✓ Cleaned arrays saved to data/processed (clean copies)
✓ Model trained successfully

Model Performance:
Training MSE: 2.6028
Test MSE: 0.7552
Training R²: 1.0000
Test R²: 1.0000
Training MAE: 0.5702
Test MAE: 0.6158


In [4]:
# Save Model and Results
print("Saving Model and Results...")

# Save the trained model
joblib.dump(ridge_model, '../../models/ridge_regression.pkl')
print("✓ Ridge Regression model saved")

# Save model performance metrics
performance_metrics = {
    'model_name': 'Ridge Regression',
    'alpha': 1.0,
    'train_mse': float(train_mse),
    'test_mse': float(test_mse),
    'train_r2': float(train_r2),
    'test_r2': float(test_r2),
    'train_mae': float(train_mae),
    'test_mae': float(test_mae),
    'n_features': len(feature_columns),
    'n_train_samples': len(X_train),
    'n_test_samples': len(X_test)
}

import json
with open('../../data/processed/ridge_performance_metrics.json', 'w') as f:
    json.dump(performance_metrics, f, indent=2)

print("✓ Performance metrics saved")
print("\nRidge Regression model training completed successfully!")


Saving Model and Results...
✓ Ridge Regression model saved
✓ Performance metrics saved

Ridge Regression model training completed successfully!
